# Fine-tune PSKUS gesture model on METC data
Starts from PSKUS best.pt and adapts to METC visual conditions.

In [ ]:
# Cell 1 — Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

In [ ]:
# Cell 2 — Unzip METC dataset
import zipfile, os

ZIP_PATH   = '/content/drive/MyDrive/handwash_runs/metc_dataset.zip'
EXTRACT_TO = '/content/'

print('Unzipping METC dataset...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_TO)

DATASET_PATH = '/content/data/gesture/dataset'
for split in ['train', 'val', 'test']:
    classes = os.listdir(f'{DATASET_PATH}/{split}')
    total = sum(len(os.listdir(f'{DATASET_PATH}/{split}/{c}')) for c in classes)
    print(f'{split}: {total} frames across {len(classes)} classes')

In [ ]:
# Cell 3 — Fine-tune from PSKUS weights on METC
from ultralytics import YOLO

PSKUS_WEIGHTS = '/content/drive/MyDrive/handwash_runs/pskus_exp2/weights/best.pt'

model = YOLO(PSKUS_WEIGHTS)

results = model.train(
    data=DATASET_PATH,
    epochs=50,
    imgsz=224,
    batch=32,
    lr0=1e-4,        # lower LR for fine-tuning
    device=0,
    project='/content/drive/MyDrive/handwash_runs',
    name='metc_finetune',
    dropout=0.2,
    flipud=0.0,
    fliplr=0.0,
    degrees=15.0,
    translate=0.1,
    scale=0.2,
)

print(f'Training complete. Results saved to {results.save_dir}')

In [ ]:
# Cell 4 — Evaluate on METC test set
import os
from collections import defaultdict
import cv2
from ultralytics import YOLO

WHO_STEP_LABELS = {
    0: 'palm_to_palm',
    1: 'palm_over_dorsum',
    2: 'fingers_interlaced',
    3: 'backs_of_fingers',
    4: 'rotational_thumb',
    5: 'fingertips_to_palm',
}
LABEL_TO_ID = {v: k for k, v in WHO_STEP_LABELS.items()}

best_weights = str(results.save_dir / 'weights' / 'best.pt')
model = YOLO(best_weights)

test_dir = f'{DATASET_PATH}/test'
correct = defaultdict(int)
total = defaultdict(int)

for label in os.listdir(test_dir):
    if label not in LABEL_TO_ID:
        continue
    true_id = LABEL_TO_ID[label]
    class_dir = f'{test_dir}/{label}'
    images = [f for f in os.listdir(class_dir) if f.endswith('.jpg')]
    print(f'Evaluating {label} ({len(images)} frames)...', flush=True)
    for img_file in images:
        frame = cv2.imread(f'{class_dir}/{img_file}')
        if frame is None:
            continue
        res = model.predict(source=frame, imgsz=224, verbose=False)
        if res and res[0].probs is not None:
            pred_id = int(res[0].probs.top1)
            total[label] += 1
            if pred_id == true_id:
                correct[label] += 1

print('\n── Per-class accuracy (METC test set) ──')
overall_c, overall_t = 0, 0
for i in range(6):
    label = WHO_STEP_LABELS[i]
    t = total[label]
    c = correct[label]
    acc = (c / t * 100) if t > 0 else 0.0
    overall_c += c
    overall_t += t
    print(f'  Step {i+1}/6  {label:<25} {c:>5}/{t:<5} {acc:5.1f}%')

print(f'\n  Overall: {overall_c}/{overall_t} = {overall_c/overall_t*100:.1f}%')

In [ ]:
# Cell 5 — Copy best weights to Drive root for easy access
import shutil

src = str(results.save_dir / 'weights' / 'best.pt')
dst = '/content/drive/MyDrive/handwash_runs/metc_finetune_best.pt'
shutil.copy2(src, dst)
print(f'Weights saved to {dst}')